In [ ]:
# %%
%pip install seaborn
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import seaborn as sns
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    make_scorer
)
import copy
from scipy.optimize import minimize
torch.set_default_dtype(torch.float32)

# %% [markdown]
# LOAD DATASET

# %%
df=pd.read_csv('train.csv')
#9 inputs 19 outputs
X_train=np.array(df.iloc[:,:9])
y_train=np.array(df.iloc[:,9:])

df=pd.read_csv('test.csv')

X_test=np.array(df.iloc[:,:9])
y_test=np.array(df.iloc[:,9:])

names=df.columns

n_inputs=X_train.shape[1]
n_outputs=y_train.shape[1]

input_names=names[:9]
output_names=names[9:]

print(output_names)

# %% [markdown]
# PLOTTING

# %%
for i in range(y_train.shape[1]):  
    fig, ax = plt.subplots(1, X_train.shape[1], figsize=(16,9), sharey=True)
    
    fig.suptitle(f'{output_names[i]}', fontsize=14, fontweight='bold')
    
    for j in range(X_train.shape[1]):
        ax[j].scatter(X_train[:,j], y_train[:,i], marker='.', alpha=0.5)
        ax[j].set_xlabel(input_names[j])
        
        if j == 0:
            ax[j].set_ylabel(output_names[i])
            
    plt.tight_layout()
    plt.show()

# %%
# --- input distributions ---
cols = 3
rows = math.ceil(X_train.shape[1] / cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, 4*rows))
fig.suptitle("Marginal Distributions of Inputs", fontsize=15, fontweight='bold')

for i in range(X_train.shape[1]):
    ax = axes.flatten()[i]
    sns.histplot(X_train[:, i], kde=True, ax=ax, color='darkblue')
    
    ax.set_title(input_names[i])
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# --- output distributions ---
cols = 3
rows = math.ceil(y_train.shape[1] / cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, 4*rows))
fig.suptitle("Marginal Distributions of Outputs", fontsize=15, fontweight='bold')
for i in range(y_train.shape[1]):
    ax= axes.flatten()[i]
    sns.histplot(y_train[:, i], kde=True, ax=ax, color='darkred')
    ax.set_title(output_names[i])  
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()
    


# %%
#Correlation heat matrix
df_X = pd.DataFrame(X_train, columns=input_names)
df_y = pd.DataFrame(y_train, columns=output_names)
df_all = pd.concat([df_X, df_y], axis=1)

corr_matrix = df_all.corr() # corretlation matrix

plt.figure(figsize=(22, 18))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title("Correlation Matrix (Inputs & Outputs)", fontsize=18)
plt.tight_layout()
plt.show()

# %%
# check if some of the variables are almost constant so that we'll be able to remove them
# from the neural network training, since they won't provide any useful information to the model
variances = df_y.var()
stdevs = df_y.std()

print("Variance of outputs:")
for out_name, var, std in zip(output_names, variances, stdevs):
    # if variance is almost zwero the output is const
    if var < 1e-4:
        print(f"⚠️ {out_name} is almost constant (Var: {var:.2e}, Std: {std:.2e})")

# %% [markdown]
# MULTILINEAR REGRESSION

# %%
X_avg=np.mean(X_train,axis=0)
X_std=np.std(X_train,axis=0)
X_train_n=(X_train-X_avg)/X_std

def AIC(y_true, y_pred, k):
    # AIC = n * log(RSS/n) + 2k
    n = len(y_true)
    rss = ((y_true - y_pred) ** 2).sum()
    return n * np.log(rss / n) + 2 * k

def BIC(y_true, y_pred, k):
    # BIC = n * log(RSS/n) + k * log(n)
    n = len(y_true)
    rss = ((y_true - y_pred) ** 2).sum()
    return n * np.log(rss / n) + k * np.log(n)


alphas = np.logspace(-4, 2, 100)
for i,output in enumerate(output_names):
    scores_aic = []
    scores_bic = []
    for alpha in alphas:
        lasso = Lasso(alpha=alpha)
        lasso.fit(X_train_n, y_train[:,i])
        y_pred = lasso.predict(X_train_n)
        k = np.sum(lasso.coef_ != 0) + 1  # non-zero features + intercept
        scores_aic.append(AIC(y_train[:,i], y_pred, k))
        scores_bic.append(BIC(y_train[:,i], y_pred, k))
        
    plt.figure(i,figsize=(16, 9))
    plt.plot(alphas, scores_aic, label='AIC', color='blue')
    plt.plot(alphas, scores_bic, label='BIC', color='red')
    #plt.plot(alphas, np.sqrt([scores_aic[j] * scores_bic[j] for j in range(len(alphas))]),
    #     label='Geometric Mean (AIC*BIC)^0.5', color='green')
    plt.xscale('log')
    plt.xlabel('Alpha (Regularization Strength)')
    plt.ylabel('Information Criteria Score')
    plt.title(f'[{output}], AIC and BIC Scores for Lasso Regression')
    plt.legend()
    plt.grid()
plt.show() 


'''
alpha=1e-3

lasso=Lasso(alpha=alpha)
lasso.fit(X_train_n,y_train[:,0])
coefficients=lasso.coef_
print(coefficients)
y_pred=lasso.predict(X_train_n)



fig,ax =plt.subplots(1,2)

ax[0].scatter(y_train[:,0],y_pred,marker='.')
ax[0].set_xlabel('True Values')
ax[0].set_ylabel('Predicted Values')
plt.show()
'''

# %%
plt.figure(figsize=(12, 6))
plt.plot(alphas, scores_aic, label='AIC', color='blue')
plt.plot(alphas, scores_bic, label='BIC', color='red')
plt.plot(alphas, np.sqrt([scores_aic[i] * scores_bic[i] for i in range(len(alphas))]),
         label='Geometric Mean (AIC*BIC)^0.5', color='green')
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('Information Criteria Score')
plt.title('AIC and BIC Scores for Lasso Regression')
plt.legend()
plt.grid()
plt.show()

# %% [markdown]
# NN TRAIN

# %% [markdown]
# NN CREATION

# %%
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42)

X_scaler=StandardScaler()
y_scaler=StandardScaler()

X_train_scaled=X_scaler.fit_transform(X_tr)
X_val_scaled=X_scaler.transform(X_val) 
X_test_scaled=X_scaler.transform(X_test)

y_train_scaled=y_scaler.fit_transform(y_tr)
y_val_scaled=y_scaler.transform(y_val) 
y_test_scaled=y_scaler.transform(y_test)

X_train_t=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_t=torch.tensor(y_train_scaled,dtype=torch.float32)

X_val_t=torch.tensor(X_val_scaled,dtype=torch.float32) 
y_val_t=torch.tensor(y_val_scaled,dtype=torch.float32) 

X_test_t=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_t=torch.tensor(y_test_scaled,dtype=torch.float32)

train_loader=DataLoader(TensorDataset(X_train_t,y_train_t), batch_size=32, shuffle=True)

# %%
net=nn.Sequential(
    nn.Linear(9,124),
    nn.Tanh(),
    nn.Linear(124,124),
    nn.Tanh(),
    nn.Linear(124,124),
    nn.Tanh(),
    nn.Linear(124,19),
)
criterion=nn.MSELoss()

optimizer=torch.optim.Adam(net.parameters())

# %%
n_epochs=50

net.train()

train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 10
epochs_no_improve = 0
best_model_weights = copy.deepcopy(net.state_dict())

history_acc=[]
for epoch in range(n_epochs):
    correct=0
    batch_losses = [] # average loss betwenn all batche
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad() #We eliminate gradients to have the batches fully independent from each other
        #['Q1_MW', 'Q2_MW', 'T_reactor_out', 'X_CO', 'T3', 'P3', 'n3_total',
        #'y_CO_3', 'y_H2_3', 'y_MeOH_3', 'y_H2O_3', 'P4', 'n4_total', 'x_CO_4',
        #'x_H2_4', 'x_MeOH_4', 'x_H2O_4', 'MeOH_recovery',
        #'MeOH_purity_stream4'] 3,7,8,9,10,13,14,15,16,17
        raw_out = net(X_batch)

        out = raw_out.clone()
        '''
        # Single variables bounded between 0 and 1
        bounded_idx = [3, 17, 18]   # X_CO, MeOH_recovery, MeOH_purity_stream4
        out[:, bounded_idx] = torch.sigmoid(raw_out[:, bounded_idx])

        # Gas molar fractions: y_CO_3, y_H2_3, y_MeOH_3, y_H2O_3
        gas_frac_idx = [7, 8, 9, 10]
        out[:, gas_frac_idx] = F.softmax(raw_out[:, gas_frac_idx], dim=1)

        # Liquid molar fractions: x_CO_4, x_H2_4, x_MeOH_4, x_H2O_4
        liquid_frac_idx = [13, 14, 15, 16]
        out[:, liquid_frac_idx] = F.softmax(raw_out[:, liquid_frac_idx], dim=1)
        '''
        
        loss=criterion(out,y_batch) #We compute the loss for each batch
        loss.backward() #We back-propagate the loss 
        optimizer.step() #We use the optimizer's algorithm to change the weights based on the loss function 
        batch_losses.append(loss.item())
        #predicted=out.argmax(dim=1)
        #correct += (predicted==y_batch).sum().item()
    
    #average loss for this epoch
    train_loss = np.mean(batch_losses)
    train_losses.append(train_loss)

    #test on the validationset at the end of this epoch
    net.eval()
    with torch.no_grad():
        val_out = net(X_val_t)
        val_loss = criterion(val_out, y_val_t).item()
    val_losses.append(val_loss)
    
    # Early stop if needed
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(net.state_dict()) 
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch}!")
        break

    net.train() # put the model back in training mode for the next epoch


net.load_state_dict(best_model_weights)

# train vs val curve
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Train vs Validation')
plt.legend()
plt.grid(True)
plt.show()

# %%
net.eval()

with torch.no_grad():
    model=net(X_train_t).numpy()
    model=y_scaler.inverse_transform(model)
# Single variables bounded between 0 and 1


'''

bounded_idx = [3, 17, 18]   # X_CO, MeOH_recovery, MeOH_purity_stream4
out[:, bounded_idx] = torch.sigmoid(model[:, bounded_idx])

# Gas molar fractions: y_CO_3, y_H2_3, y_MeOH_3, y_H2O_3
gas_frac_idx = [7, 8, 9, 10]
out[:, gas_frac_idx] = F.softmax(model[:, gas_frac_idx], dim=1)

# Liquid molar fractions: x_CO_4, x_H2_4, x_MeOH_4, x_H2O_4
liquid_frac_idx = [13, 14, 15, 16]
out[:, liquid_frac_idx] = F.softmax(model[:, liquid_frac_idx], dim=1)
'''
#mask=y_train !=0
#train_error=abs((model[mask]-y_train[mask])/y_train[mask])*100
#print(train_error.shape)

# %%
#print(np.average(train_error,axis=0))

for i in range(y_tr.shape[1]):

    y_true = y_tr[:, i]
    y_pred = model[:, i]

    # avoid division by zero
    mask = np.abs(y_true) > 1e-12

    if mask.sum() == 0:
        print(f"Skipping {output_names[i]} because all values are zero.")
        continue

    train_error = np.abs((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100

    plt.figure(figsize=(16, 6))
    plt.title(f'NN error for: {output_names[i]}')
    plt.scatter(range(train_error.shape[0]), train_error, s=10)
    plt.grid(True)
    plt.xlabel('Sample index')
    plt.ylabel('Error [%]')
    plt.show()

# %% [markdown]
# POLYNOMINAL RIDGE REGRESSION 
# 

# %%
# Data set preparaation for baseline model, I removed the train-test split since now there is also abpve in the NN
# this way they're using the same data and the comparison is more fair, otherwise the baseline would be trained on less data and it would be a bit unfair

X_full = X_train.copy()
y_full = y_train.copy()

# Customiazation of the evaluation metric:
# We are creating a custom scorer for GridSearchCV that computes the mean normalized RMSE across all output variables.
# The normalization is done by dividing the RMSE of each output by the range of that output in the training set.

def mean_normalized_rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))

    y_range = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    y_range = np.where(y_range == 0, 1.0, y_range)  # Avoid division by zero for constant outputs

    nrmse = rmse / y_range

    # GridSearchCV maximizes the score, therefore return negative error
    return -np.mean(nrmse)

nrmse_scorer = make_scorer(
    mean_normalized_rmse,
    greater_is_better=True
) 

# Prepare names
output_names_list = list(output_names)
input_names_list = list(input_names)


# Table: number of polynomial features for each degree
degrees_to_check = [1, 2, 3]
feature_count_rows = []
for degree in degrees_to_check:
    poly_tmp = PolynomialFeatures(degree=degree, include_bias=False)
    poly_tmp.fit(np.zeros((1, n_inputs)))

    feature_count_rows.append({
        "Degree": degree,
        "Number of polynomial features": poly_tmp.n_output_features_
    })
feature_count_table = pd.DataFrame(feature_count_rows)

print("Feature count per degree:")
display(feature_count_table)


# Polynomial Ridge pipeline
# We are introducing pipeline to creat automatic order of actions
# StandartScaler is standartizing the inputs
# PolinomialFeatures is creating the polynomial features (combinations of the original input parameters)
# Ridge is regularizing the regression

base_model = Pipeline([
    ("x_scaler", StandardScaler()),
    ("poly", PolynomialFeatures(include_bias=False)),
    ("ridge", Ridge())
])

# TransformedTargetRegressor is standardizing the outputs
poly_ridge_model = TransformedTargetRegressor(
    regressor=base_model,
    transformer=StandardScaler()
)

# Hyperparameter grid

alphas = np.logspace(-4, 2, 100)
param_grid = {
    "regressor__poly__degree": [1, 2, 3],
    "regressor__ridge__alpha": alphas
}


# GridSearch with cross-validation 
# Employed to find the best combination of polynomial degree and ridge regularization strength
# CV is set to 5 folds of cross validation
# Scoring is done with custo mean NRMSE 

grid = GridSearchCV(
    estimator=poly_ridge_model,
    param_grid=param_grid,
    scoring=nrmse_scorer,
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_tr, y_tr)

print("Best parameters:")
print(grid.best_params_)

print("\nBest CV score:")
print(grid.best_score_)

# Validation set evaluation

best_poly_ridge = grid.best_estimator_
y_val_pred = best_poly_ridge.predict(X_val)

mae_val = mean_absolute_error(
    y_val,
    y_val_pred,
    multioutput="raw_values"
)

rmse_val = np.sqrt(mean_squared_error(
    y_val,
    y_val_pred,
    multioutput="raw_values"
))

r2_val = r2_score(
    y_val,
    y_val_pred,
    multioutput="raw_values"
)


# Retrain final model on full train.csv

best_degree = grid.best_params_["regressor__poly__degree"]
best_alpha = grid.best_params_["regressor__ridge__alpha"]

final_poly_ridge = TransformedTargetRegressor(
    regressor=Pipeline([
        ("x_scaler", StandardScaler()),
        ("poly", PolynomialFeatures(
            degree=best_degree,
            include_bias=False
        )),
        ("ridge", Ridge(alpha=best_alpha))
    ]),
    transformer=StandardScaler()
)

final_poly_ridge.fit(X_full, y_full)

print("\nFinal Polynomial Ridge model trained on full train.csv")
print("Polynomial degree:", best_degree)
print("Ridge alpha:", best_alpha)


# Final test set evaluation
y_test_pred = final_poly_ridge.predict(X_test)

mae_test = mean_absolute_error(
    y_test,
    y_test_pred,
    multioutput="raw_values"
)

rmse_test = np.sqrt(mean_squared_error(
    y_test,
    y_test_pred,
    multioutput="raw_values"
))

r2_test = r2_score(
    y_test,
    y_test_pred,
    multioutput="raw_values"
)


# Combined validation and test results table
combined_results = pd.DataFrame({
    "Output": output_names,
    "MAE_val": mae_val,
    "MAE_test": mae_test,
    "RMSE_val": rmse_val,
    "RMSE_test": rmse_test,
    "R2_val": r2_val,
    "R2_test": r2_test
})

combined_results_display = combined_results.copy()

for col in ["MAE_val", "MAE_test", "RMSE_val", "RMSE_test"]:
    combined_results_display[col] = combined_results_display[col].map(lambda x: float(f"{x:.6g}"))

for col in ["R2_val", "R2_test"]:
    combined_results_display[col] = combined_results_display[col].map(lambda x: float(f"{x:.5f}"))

print("\nCombined validation and test results:")
display(combined_results_display)


# Average metrics table
average_metrics = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "Mean MAE": [np.mean(mae_val), np.mean(mae_test)],
    "Mean RMSE": [np.mean(rmse_val), np.mean(rmse_test)],
    "Mean R2": [np.mean(r2_val), np.mean(r2_test)],
    "Median R2": [np.median(r2_val), np.median(r2_test)],
    "Minimum R2": [np.min(r2_val), np.min(r2_test)]
})

average_metrics_display = average_metrics.copy()

for col in ["Mean MAE", "Mean RMSE"]:
    average_metrics_display[col] = average_metrics_display[col].map(lambda x: float(f"{x:.6g}"))

for col in ["Mean R2", "Median R2", "Minimum R2"]:
    average_metrics_display[col] = average_metrics_display[col].map(lambda x: float(f"{x:.5f}"))

print("\nAverage metrics:")
display(average_metrics_display)


# Table with all polynomial features for the best degree
poly_best = PolynomialFeatures(
    degree=best_degree,
    include_bias=False
)

poly_best.fit(np.zeros((1, n_inputs)))
best_feature_names = poly_best.get_feature_names_out(input_names_list)

all_features_table = pd.DataFrame({
    "Feature index": np.arange(1, len(best_feature_names) + 1),
    "Feature name": best_feature_names
})

print(f"\nAll polynomial features for best degree = {best_degree}:")
display(all_features_table)


# Cross-validation curve
cv_results = pd.DataFrame(grid.cv_results_)
plt.figure(figsize=(12, 6))

degree_colors = {
    1: "blue",
    2: "red",
    3: "green"
}

for degree in sorted(cv_results["param_regressor__poly__degree"].unique()):

    subset = cv_results[
        cv_results["param_regressor__poly__degree"] == degree
    ].copy()

    subset["alpha"] = subset["param_regressor__ridge__alpha"].astype(float)
    subset = subset.sort_values("alpha")

    alphas_plot = subset["alpha"].to_numpy()
    cv_nrmse = (-subset["mean_test_score"]).to_numpy()

    plt.plot(
        alphas_plot,
        cv_nrmse,
        color=degree_colors[int(degree)],
        linewidth=2,
        label=f"Degree = {degree}"
    )

plt.xscale("log")

# Major ticks only, evenly spaced in log scale
major_ticks = [1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2]
plt.xticks(major_ticks)
plt.xlabel("Alpha (Regularization Strength)")
plt.ylabel("Mean CV Normalized RMSE")
plt.title("Cross-Validation Scores for Polynomial Ridge Regression")
plt.grid(True, which="major", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# R2 comparison: validation vs test
x_pos = np.arange(len(output_names_list))
width = 0.38
plt.figure(figsize=(14, 6))
plt.bar(
    x_pos - width / 2,
    r2_val,
    width=width,
    label="Validation R2"
)
plt.bar(
    x_pos + width / 2,
    r2_test,
    width=width,
    label="Test R2"
)
plt.xticks(x_pos, output_names_list, rotation=90)
plt.ylabel("R2")
plt.title("Polynomial Ridge R2 per Output")
plt.ylim(0.90, 1.01)
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.show()


# RMSE comparison: validation vs test
plt.figure(figsize=(14, 6))
plt.bar(
    x_pos - width / 2,
    rmse_val,
    width=width,
    label="Validation RMSE"
)
plt.bar(
    x_pos + width / 2,
    rmse_test,
    width=width,
    label="Test RMSE"
)
plt.xticks(x_pos, output_names_list, rotation=90)
plt.ylabel("RMSE")
plt.title("Polynomial Ridge RMSE per Output")
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.show()

#  MAE comparison: validation vs test
plt.figure(figsize=(14, 6))
plt.bar(
    x_pos - width / 2,
    mae_val,
    width=width,
    label="Validation MAE"
)
plt.bar(
    x_pos + width / 2,
    mae_test,
    width=width,
    label="Test MAE"
)
plt.xticks(x_pos, output_names_list, rotation=90)
plt.ylabel("MAE")
plt.title("Polynomial Ridge MAE per Output")
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.show()

# Parity plots for all outputs
n_cols = 4
n_rows = int(np.ceil(n_outputs / n_cols))
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.5 * n_cols, 4.5 * n_rows)
)
axes = axes.flatten()
for i, output in enumerate(output_names_list):
    ax = axes[i]
    y_true = y_test[:, i]
    y_pred = y_test_pred[:, i]
    ax.scatter(y_true, y_pred, s=8, alpha=0.45)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot(
        [min_val, max_val],
        [min_val, max_val],
        "k--",
        linewidth=1.5
    )
    ax.set_title(output)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.grid(True, alpha=0.4)
for j in range(n_outputs, len(axes)):
    axes[j].axis("off")
fig.suptitle("Polynomial Ridge Parity Plots for Outputs", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


# Residual plots vs predicted value for all outputs
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.5 * n_cols, 4.5 * n_rows)
)
axes = axes.flatten()
for i, output in enumerate(output_names_list):
    ax = axes[i]
    y_true = y_test[:, i]
    y_pred = y_test_pred[:, i]
    residual = y_pred - y_true
    ax.scatter(y_pred, residual, s=8, alpha=0.45)
    ax.axhline(0, color="k", linestyle="--", linewidth=1.5)
    ax.set_title(output)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual")
    ax.grid(True, alpha=0.4)
for j in range(n_outputs, len(axes)):
    axes[j].axis("off")
fig.suptitle("Polynomial Ridge Residuals vs Predicted Values", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# Residual plots vs T_flash for all outputs
if "T_flash" in input_names_list:
    T_flash_idx = input_names_list.index("T_flash")
else:
    T_flash_idx = 7
T_flash_test = X_test[:, T_flash_idx]
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.5 * n_cols, 4.5 * n_rows)
)
axes = axes.flatten()
for i, output in enumerate(output_names_list):
    ax = axes[i]
    y_true = y_test[:, i]
    y_pred = y_test_pred[:, i]
    residual = y_pred - y_true
    ax.scatter(T_flash_test, residual, s=8, alpha=0.45)
    ax.axhline(0, color="k", linestyle="--", linewidth=1.5)
    ax.set_title(output)
    ax.set_xlabel("T_flash")
    ax.set_ylabel("Residual")
    ax.grid(True, alpha=0.4)
for j in range(n_outputs, len(axes)):
    axes[j].axis("off")
fig.suptitle("Polynomial Ridge Residuals vs T_flash", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# %% [markdown]
# HYBRID MODEL 
# 

# %%
# Data set preparation for hybrid model

X_full = X_train.copy()
y_full = y_train.copy()

X_tr, X_val, y_tr, y_val = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

output_names_list = list(output_names)
input_names_list = list(input_names)


# Customization of the evaluation metric:
# We are creating a custom scorer for GridSearchCV that computes the mean normalized RMSE across all output variables.
# The normalization is done by dividing the RMSE of each output by the range of that output.

def mean_normalized_rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))

    y_range = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    y_range = np.where(y_range == 0, 1.0, y_range)  # Avoid division by zero for constant outputs

    nrmse = rmse / y_range

    # GridSearchCV maximizes the score, therefore return negative error
    return -np.mean(nrmse)


nrmse_scorer = make_scorer(
    mean_normalized_rmse,
    greater_is_better=True
)


# Polynomial Ridge part of the hybrid architecture
# The Polynomial Ridge model is used to capture the main smooth nonlinear trend of the process.
# Later, the neural network will be trained only on the residual errors of this Polynomial Ridge model.

poly_base_model = Pipeline([
    ("x_scaler", StandardScaler()),
    ("poly", PolynomialFeatures(include_bias=False)),
    ("ridge", Ridge())
])

poly_ridge_model = TransformedTargetRegressor(
    regressor=poly_base_model,
    transformer=StandardScaler()
)


# Hyperparameter grid for Polynomial Ridge

alphas = np.logspace(-4, 2, 100)

param_grid = {
    "regressor__poly__degree": [1, 2, 3],
    "regressor__ridge__alpha": alphas
}


# GridSearch with cross-validation for Polynomial Ridge
# Employed to find the best combination of polynomial degree and ridge regularization strength.
# CV is set to 5 folds of cross-validation.
# Scoring is done with custom mean normalized RMSE.

grid = GridSearchCV(
    estimator=poly_ridge_model,
    param_grid=param_grid,
    scoring=nrmse_scorer,
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_tr, y_tr)

best_poly_model = grid.best_estimator_

best_degree = grid.best_params_["regressor__poly__degree"]
best_alpha = grid.best_params_["regressor__ridge__alpha"]

print("Best Polynomial Ridge parameters:")
print("Polynomial degree:", best_degree)
print("Ridge alpha:", best_alpha)

print("\nBest Polynomial Ridge CV score:")
print(grid.best_score_)


# Polynomial Ridge predictions
# These predictions represent the baseline part of the hybrid architecture.

y_poly_tr = best_poly_model.predict(X_tr)
y_poly_val = best_poly_model.predict(X_val)
y_poly_test = best_poly_model.predict(X_test)


# Residual target calculation
# The neural network is not trained to predict the full output directly.
# Instead, it is trained to predict the residual error of the Polynomial Ridge model.

res_tr = y_tr - y_poly_tr
res_val = y_val - y_poly_val

print("\nResidual target shape:")
print("Training residuals:", res_tr.shape)
print("Validation residuals:", res_val.shape)


# Scaling for neural network
# Inputs are standardized for neural network training.
# Residuals are also standardized because residuals of different outputs may have different numerical scales.

x_scaler_nn = StandardScaler()
res_scaler_nn = StandardScaler()

X_tr_scaled = x_scaler_nn.fit_transform(X_tr)
X_val_scaled = x_scaler_nn.transform(X_val)
X_test_scaled = x_scaler_nn.transform(X_test)

res_tr_scaled = res_scaler_nn.fit_transform(res_tr)
res_val_scaled = res_scaler_nn.transform(res_val)


# PyTorch tensor preparation

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

X_tr_tensor = torch.tensor(X_tr_scaled, dtype=torch.float32)
res_tr_tensor = torch.tensor(res_tr_scaled, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
res_val_tensor = torch.tensor(res_val_scaled, dtype=torch.float32).to(device)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

train_dataset = TensorDataset(
    X_tr_tensor,
    res_tr_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)


# Neural network correction model
# This network learns the residual correction:
# residual = y_true - y_poly
# Final hybrid prediction:
# y_hybrid = y_poly + residual_NN

class ResidualCorrectionNN(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_inputs, 64),
            nn.SiLU(),

            nn.Linear(64, 64),
            nn.SiLU(),

            nn.Linear(64, n_outputs)
        )

    def forward(self, x):
        return self.net(x)


residual_nn = ResidualCorrectionNN(
    n_inputs=n_inputs,
    n_outputs=n_outputs
).to(device)

print("\nResidual correction neural network:")
print(residual_nn)


# Neural network training setup
# MSELoss is applied to scaled residuals.
# AdamW is used as optimizer with a small weight decay for regularization.
# ReduceLROnPlateau decreases learning rate when validation loss stops improving.
# Early stopping avoids overfitting.

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    residual_nn.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=15
)

max_epochs = 1000
early_stopping_patience = 50

best_val_loss = np.inf
best_model_state = None
epochs_without_improvement = 0

train_loss_history = []
val_loss_history = []


# Training loop for neural residual correction model

for epoch in range(max_epochs):

    residual_nn.train()
    batch_losses = []

    for X_batch, res_batch in train_loader:

        X_batch = X_batch.to(device)
        res_batch = res_batch.to(device)

        optimizer.zero_grad()

        res_pred_batch = residual_nn(X_batch)
        loss = criterion(res_pred_batch, res_batch)

        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())

    train_loss = np.mean(batch_losses)

    residual_nn.eval()

    with torch.no_grad():
        res_val_pred_scaled = residual_nn(X_val_tensor)
        val_loss = criterion(res_val_pred_scaled, res_val_tensor).item()

    scheduler.step(val_loss)

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in residual_nn.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1:4d} | "
            f"Train loss: {train_loss:.6e} | "
            f"Val loss: {val_loss:.6e}"
        )

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break


# Load best neural network weights

residual_nn.load_state_dict(best_model_state)

print("\nBest validation loss for residual correction NN:")
print(best_val_loss)


# Training history plot

plt.figure(figsize=(9, 5))

plt.plot(
    train_loss_history,
    label="Training loss",
    linewidth=2
)

plt.plot(
    val_loss_history,
    label="Validation loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("MSE loss on scaled residuals")
plt.title("Hybrid Model: Residual Neural Network Training History")
plt.grid(True, alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


# Neural network residual prediction
# The neural network predicts scaled residuals.
# Then the residuals are transformed back to original physical units.

residual_nn.eval()

with torch.no_grad():
    res_val_pred_scaled = residual_nn(X_val_tensor).cpu().numpy()
    res_test_pred_scaled = residual_nn(X_test_tensor).cpu().numpy()

res_val_pred = res_scaler_nn.inverse_transform(res_val_pred_scaled)
res_test_pred = res_scaler_nn.inverse_transform(res_test_pred_scaled)


# Final hybrid predictions
# Final hybrid prediction is obtained by adding the NN residual correction to the Polynomial Ridge prediction.

y_hybrid_val_pred = y_poly_val + res_val_pred
y_hybrid_test_pred = y_poly_test + res_test_pred


# Validation set evaluation for hybrid model

mae_val = mean_absolute_error(
    y_val,
    y_hybrid_val_pred,
    multioutput="raw_values"
)

rmse_val = np.sqrt(mean_squared_error(
    y_val,
    y_hybrid_val_pred,
    multioutput="raw_values"
))

r2_val = r2_score(
    y_val,
    y_hybrid_val_pred,
    multioutput="raw_values"
)


# Final test set evaluation for hybrid model

mae_test = mean_absolute_error(
    y_test,
    y_hybrid_test_pred,
    multioutput="raw_values"
)

rmse_test = np.sqrt(mean_squared_error(
    y_test,
    y_hybrid_test_pred,
    multioutput="raw_values"
))

r2_test = r2_score(
    y_test,
    y_hybrid_test_pred,
    multioutput="raw_values"
)


# Combined validation and test results table

hybrid_results = pd.DataFrame({
    "Output": output_names,
    "MAE_val": mae_val,
    "MAE_test": mae_test,
    "RMSE_val": rmse_val,
    "RMSE_test": rmse_test,
    "R2_val": r2_val,
    "R2_test": r2_test
})

hybrid_results_display = hybrid_results.copy()

for col in ["MAE_val", "MAE_test", "RMSE_val", "RMSE_test"]:
    hybrid_results_display[col] = hybrid_results_display[col].map(lambda x: float(f"{x:.6g}"))

for col in ["R2_val", "R2_test"]:
    hybrid_results_display[col] = hybrid_results_display[col].map(lambda x: float(f"{x:.5f}"))

print("\nHybrid model combined validation and test results:")
display(hybrid_results_display)


# Average metrics table

hybrid_average_metrics = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "Mean MAE": [np.mean(mae_val), np.mean(mae_test)],
    "Mean RMSE": [np.mean(rmse_val), np.mean(rmse_test)],
    "Mean R2": [np.mean(r2_val), np.mean(r2_test)],
    "Median R2": [np.median(r2_val), np.median(r2_test)],
    "Minimum R2": [np.min(r2_val), np.min(r2_test)]
})

hybrid_average_metrics_display = hybrid_average_metrics.copy()

for col in ["Mean MAE", "Mean RMSE"]:
    hybrid_average_metrics_display[col] = hybrid_average_metrics_display[col].map(lambda x: float(f"{x:.6g}"))

for col in ["Mean R2", "Median R2", "Minimum R2"]:
    hybrid_average_metrics_display[col] = hybrid_average_metrics_display[col].map(lambda x: float(f"{x:.5f}"))

print("\nHybrid model average metrics:")
display(hybrid_average_metrics_display)


# R2 comparison: validation vs test

x_pos = np.arange(len(output_names_list))
width = 0.38

plt.figure(figsize=(14, 6))

plt.bar(
    x_pos - width / 2,
    r2_val,
    width=width,
    label="Validation R2"
)

plt.bar(
    x_pos + width / 2,
    r2_test,
    width=width,
    label="Test R2"
)

plt.xticks(x_pos, output_names_list, rotation=90)
plt.ylabel("R2")
plt.title("Hybrid Polynomial Ridge + Neural Correction: R2 per Output")
plt.ylim(0.90, 1.01)
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


# RMSE comparison: validation vs test

plt.figure(figsize=(14, 6))

plt.bar(
    x_pos - width / 2,
    rmse_val,
    width=width,
    label="Validation RMSE"
)

plt.bar(
    x_pos + width / 2,
    rmse_test,
    width=width,
    label="Test RMSE"
)

plt.xticks(x_pos, output_names_list, rotation=90)
plt.ylabel("RMSE")
plt.title("Hybrid Polynomial Ridge + Neural Correction: RMSE per Output")
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


# MAE comparison: validation vs test

plt.figure(figsize=(14, 6))

plt.bar(
    x_pos - width / 2,
    mae_val,
    width=width,
    label="Validation MAE"
)

plt.bar(
    x_pos + width / 2,
    mae_test,
    width=width,
    label="Test MAE"
)

plt.xticks(x_pos, output_names_list, rotation=90)
plt.ylabel("MAE")
plt.title("Hybrid Polynomial Ridge + Neural Correction: MAE per Output")
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

# %% [markdown]
# COMPARISON BETWEEN THE DIFFERENT NN ARCHITECTURES

# %%
# MODEL COMPARISON: DIRECT NN vs HYBRID MODEL

output_names_list = list(output_names)
input_names_list = list(input_names)

out_idx = {name: i for i, name in enumerate(output_names_list)}

# Outputs that are most important for the optimization task
optimization_outputs = [
    "Q1_MW",
    "Q2_MW",
    "MeOH_recovery",
    "MeOH_purity_stream4"
]

optimization_idx = [out_idx[name] for name in optimization_outputs]


def predict_direct_nn_batch(X):
    """
    Prediction of the first direct neural network.
    Input: X in physical units, shape (n_samples, 9)
    Output: y_pred in physical units, shape (n_samples, 19)
    """
    net.eval()

    X_scaled = X_scaler.transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    with torch.no_grad():
        y_scaled = net(X_tensor).cpu().numpy()

    y_pred = y_scaler.inverse_transform(y_scaled)

    return y_pred


def predict_hybrid_batch(X):
    """
    Prediction of the hybrid model:
    y_hybrid = Polynomial Ridge prediction + NN residual correction.
    Input: X in physical units, shape (n_samples, 9)
    Output: y_pred in physical units, shape (n_samples, 19)
    """
    residual_nn.eval()

    # Polynomial Ridge prediction
    y_poly = best_poly_model.predict(X)

    # Residual NN prediction
    X_scaled = x_scaler_nn.transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        res_scaled = residual_nn(X_tensor).cpu().numpy()

    res_pred = res_scaler_nn.inverse_transform(res_scaled)

    y_pred = y_poly + res_pred

    return y_pred


def compute_model_summary(y_true, y_pred, model_name):
    """
    Computes global metrics and optimization-relevant metrics.
    """
    mae = mean_absolute_error(
        y_true,
        y_pred,
        multioutput="raw_values"
    )

    rmse = np.sqrt(mean_squared_error(
        y_true,
        y_pred,
        multioutput="raw_values"
    ))

    r2 = r2_score(
        y_true,
        y_pred,
        multioutput="raw_values"
    )

    y_range = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    y_range = np.where(y_range == 0, 1.0, y_range)

    nrmse = rmse / y_range

    summary = {
        "Model": model_name,

        "Mean MAE": np.mean(mae),
        "Mean RMSE": np.mean(rmse),
        "Mean NRMSE": np.mean(nrmse),
        "Mean R2": np.mean(r2),
        "Median R2": np.median(r2),
        "Minimum R2": np.min(r2),

        "Optimization outputs Mean MAE": np.mean(mae[optimization_idx]),
        "Optimization outputs Mean RMSE": np.mean(rmse[optimization_idx]),
        "Optimization outputs Mean NRMSE": np.mean(nrmse[optimization_idx]),
        "Optimization outputs Mean R2": np.mean(r2[optimization_idx]),
    }

    return summary, mae, rmse, nrmse, r2


# Predictions on validation set
y_val_pred_direct_nn = predict_direct_nn_batch(X_val)
y_val_pred_hybrid = predict_hybrid_batch(X_val)

# Predictions on test set
y_test_pred_direct_nn = predict_direct_nn_batch(X_test)
y_test_pred_hybrid = predict_hybrid_batch(X_test)


# Validation metrics
direct_val_summary, direct_val_mae, direct_val_rmse, direct_val_nrmse, direct_val_r2 = compute_model_summary(
    y_val,
    y_val_pred_direct_nn,
    "Direct NN"
)

hybrid_val_summary, hybrid_val_mae, hybrid_val_rmse, hybrid_val_nrmse, hybrid_val_r2 = compute_model_summary(
    y_val,
    y_val_pred_hybrid,
    "Hybrid Polynomial Ridge + Residual NN"
)


# Test metrics
direct_test_summary, direct_test_mae, direct_test_rmse, direct_test_nrmse, direct_test_r2 = compute_model_summary(
    y_test,
    y_test_pred_direct_nn,
    "Direct NN"
)

hybrid_test_summary, hybrid_test_mae, hybrid_test_rmse, hybrid_test_nrmse, hybrid_test_r2 = compute_model_summary(
    y_test,
    y_test_pred_hybrid,
    "Hybrid Polynomial Ridge + Residual NN"
)


comparison_table = pd.DataFrame([
    {"Dataset": "Validation",
        **direct_val_summary},
    {"Dataset": "Validation",
        **hybrid_val_summary},
    { "Dataset": "Test",
        **direct_test_summary},
    {"Dataset": "Test",
        **hybrid_test_summary}
])

comparison_display = comparison_table.copy()

for col in comparison_display.columns:
    if col not in ["Dataset", "Model"]:
        comparison_display[col] = comparison_display[col].map(lambda x: float(f"{x:.6g}"))

print("Direct NN vs Hybrid Model comparison:")
display(comparison_display)

# %%
# SELECT BEST MODEL FOR OPTIMIZATION

validation_only = comparison_table[
    comparison_table["Dataset"] == "Validation"
].copy()

# Selection criterion:
# lower NRMSE on the outputs used by the optimization problem
best_model_row = validation_only.sort_values(
    "Optimization outputs Mean NRMSE",
    ascending=True
).iloc[0]

best_model_name = best_model_row["Model"]

print("Best model selected for optimization:")
print(best_model_name)

print("\nSelection criterion:")
print("Minimum Validation Optimization outputs Mean NRMSE")
print(best_model_row["Optimization outputs Mean NRMSE"])

# %% [markdown]
# OPTIMIZATION

# %%

Qmax = 1.7          # MW
purity_min = 0.95   # purity threshold, do we impose it?

#conversion to physical units
input_names_list = list(input_names)
output_names_list = list(output_names)

out_idx = {name: i for i, name in enumerate(output_names_list)}
in_idx = {name: i for i, name in enumerate(input_names_list)}
idx_Q1 = out_idx["Q1_MW"]
idx_Q2 = out_idx["Q2_MW"]
idx_recovery = out_idx["MeOH_recovery"]
idx_purity = out_idx["MeOH_purity_stream4"]
idx_n4 = out_idx["n4_total"]
idx_xMeOH4 = out_idx["x_MeOH_4"]
idx_Tflash = in_idx["T_flash"]
idx_T4 = in_idx["T4"]

# Input bounds
bounds = [
    (800, 1600),    # n1_total
    (0.25, 0.45),   # y_CO_1
    (0.00, 0.04),   # y_H2O_1
    (0, 400),       # n2_total
    (290, 320),     # T1
    (470, 540),     # T_reactor_in
    (50, 80),       # P_R
    (290, 360),     # T_flash
    (280, 340),     # T4
]

# predict the outputs for the test set
# net.eval()
#def predict_nn_physical(x):
 #   """
 #   x: array with shape (9,)
 #   returns y_pred in physical units with shape (19,)
 #   """
 #   x = np.asarray(x, dtype=np.float32).reshape(1, -1)
 #   
 #   x_scaled = X_scaler.transform(x)
 #   x_t = torch.tensor(x_scaled, dtype=torch.float32)
 #   
 #   with torch.no_grad(): # we don't need gradients for prediction, it's not an optimization step so we can disable them for speed
 #       y_scaled = net(x_t).cpu().numpy() # predict in scaled units
 #   
 #   y_physical = y_scaler.inverse_transform(y_scaled)
 #   return y_physical[0]

def predict_surrogate_physical(x):
    """
    Uses the best selected surrogate model.
    x: array with shape (9,)
    returns y_pred in physical units with shape (19,)
    """
    x = np.asarray(x, dtype=np.float32).reshape(1, -1)

    if best_model_name == "Direct NN":
        y_pred = predict_direct_nn_batch(x)

    elif best_model_name == "Hybrid Polynomial Ridge + Residual NN":
        y_pred = predict_hybrid_batch(x)

    else:
        raise ValueError(f"Unknown model selected: {best_model_name}")

    return y_pred[0]


def predicted_kpis(x):
    # y = predict_nn_physical(x)
    y = predict_surrogate_physical(x)

    Q1 = y[idx_Q1]
    Q2 = y[idx_Q2]
    energy = abs(Q1) + abs(Q2)
    recovery = y[idx_recovery]
    purity = y[idx_purity]
    n4 = y[idx_n4] #for now we just return n4 but we could easily compute other KPIs if needed
    
    return {
        "y": y,
        "Q1": Q1,
        "Q2": Q2,
        "energy": energy,
        "recovery": recovery,
        "purity": purity,
        "n4_total": n4
    }



# objective function and costraints 
def objective(x):
    """
    scipy minimizes, therefore we minimize -MeOH_recovery.
    """
    kpis = predicted_kpis(x)
    return -kpis["recovery"] # we want to maximize recovery, so we minimize its negative

def constraint_energy(x):
    """
    Must be >= 0 for SLSQP:
    Qmax - (|Q1| + |Q2|) >= 0
    """
    kpis = predicted_kpis(x)
    return Qmax - kpis["energy"]

def constraint_purity(x):
    """
    Must be >= 0:
    MeOH_purity_stream4 - purity_min >= 0
    """
    kpis = predicted_kpis(x)
    return kpis["purity"] - purity_min

def constraint_T4_flash(x): # He2 is necesseraly a cooler it can't heat up
    """
    Must be >= 0:
    T_flash - T4 - 2 >= 0
    because T4 <= T_flash - 2
    """
    return x[idx_Tflash] - x[idx_T4] - 2.0

constraints = [
    {"type": "ineq", "fun": constraint_energy},
    {"type": "ineq", "fun": constraint_purity},
    {"type": "ineq", "fun": constraint_T4_flash},
]


# Use real training rows to find reasonable initial guesses. This avoids starting SLSQP in completely bad regions.
energy_train = np.abs(y_train[:, idx_Q1]) + np.abs(y_train[:, idx_Q2]) # compute the energy for each training point
feasible_train = (
    (energy_train <= Qmax) &
    (y_train[:, idx_purity] >= purity_min) &
    (X_train[:, idx_T4] <= X_train[:, idx_Tflash] - 2.0)
) # check which training points are feasible according to our constraints

x0_candidates = [] # list of initial guesses for the optimization

if feasible_train.sum() > 0:
    feasible_indices = np.where(feasible_train)[0]
    best_feasible_idx = feasible_indices[
        np.argmax(y_train[feasible_indices, idx_recovery]) # among the feasible points, find the one with the highest recovery to use as a starting point
    ]
    x0_candidates.append(X_train[best_feasible_idx].copy())

    # Add other good feasible points
    sorted_feasible = feasible_indices[
        np.argsort(-y_train[feasible_indices, idx_recovery]) # needed the minus sign to sort in descending order
    ]
    for idx in sorted_feasible[:10]:
        x0_candidates.append(X_train[idx].copy())

else:
    print("Warning: no feasible training point found for the selected purity_min.")

    #check how far from the constrain the pts are
    energy_violation = np.maximum(energy_train - Qmax, 0.0) / Qmax
    purity_violation = np.maximum(purity_min - y_train[:, idx_purity], 0.0)

    score = (
        y_train[:, idx_recovery]
        - 10.0 * energy_violation
        - 10.0 * purity_violation
    ) # i could change the weights, for now put 10 cause it's heavy and I know there are feasible pts

    
    #best_indices = np.argsort(-score)[:10]
    #for idx in best_indices:
        #x0_candidates.append(X_train[idx].copy())


# Add random starting points inside the input domain
rng = np.random.default_rng(42)

low = np.array([b[0] for b in bounds])
high = np.array([b[1] for b in bounds])

random_points = rng.uniform(low, high, size=(40, len(bounds)))

# imposeT4 constraint on random initial guesses
random_points[:, idx_T4] = np.minimum(
    random_points[:, idx_T4],
    random_points[:, idx_Tflash] - 2.0
)

random_points[:, idx_T4] = np.clip(
    random_points[:, idx_T4],
    bounds[idx_T4][0],
    bounds[idx_T4][1]
)

for x0 in random_points:
    x0_candidates.append(x0.copy())

print(f"Number of starting points: {len(x0_candidates)}")

solutions = []

for k, x0 in enumerate(x0_candidates):
    res = minimize(
        objective,
        x0=x0,
        method="SLSQP", #we need a method that can handle non-linear constraints, SLSQP is a good choice for this type of problem
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": 500,
            "ftol": 1e-9,
            "disp": False
        }
    )
    
    x_opt = res.x
    kpis = predicted_kpis(x_opt)
    
    # check on the optimal point
    energy_slack = Qmax - kpis["energy"]
    purity_slack = kpis["purity"] - purity_min
    T4_slack = x_opt[idx_Tflash] - x_opt[idx_T4] - 2.0
    
    feasible = (
        energy_slack >= -1e-6 and 
        purity_slack >= -1e-6 and
        T4_slack >= -1e-6
    )
    
    solutions.append({
        "start_id": k,
        "success": res.success,
        "message": res.message,
        "feasible": feasible,
        "objective": res.fun,
        "x": x_opt,
        "y": kpis["y"],
        "recovery": kpis["recovery"],
        "purity": kpis["purity"],
        "energy": kpis["energy"],
        "Q1": kpis["Q1"],
        "Q2": kpis["Q2"],
        "n4_total": kpis["n4_total"],
        "energy_slack": energy_slack,
        "purity_slack": purity_slack,
        "T4_slack": T4_slack,
    })


feasible_solutions = [s for s in solutions if s["feasible"]] # only solutions which respet the constraints
if len(feasible_solutions) == 0:
    print("No feasible solution found.")
    print("Showing the least-violating solution instead.")

else:
    best_sol = max(feasible_solutions, key=lambda s: s["recovery"])

x_opt = best_sol["x"]
y_opt = best_sol["y"]

print("\nBest solution found")
print("-------------------")
print("Optimizer success:", best_sol["success"])
print("Feasible:", best_sol["feasible"])
print("Predicted MeOH recovery:", best_sol["recovery"])
print("Predicted MeOH purity:", best_sol["purity"])
print("Predicted total energy |Q1| + |Q2| [MW]:", best_sol["energy"])


opt_inputs = pd.DataFrame({
    "Input": input_names_list,
    "Optimal value": x_opt
})

print("\nOptimal operating conditions:")
display(opt_inputs)

important_outputs = [
    "Q1_MW",
    "Q2_MW",
    "T_reactor_out",
    "X_CO",
    "n3_total",
    "n4_total",
    "x_MeOH_4",
    "x_H2O_4",
    "MeOH_recovery",
    "MeOH_purity_stream4"
]

opt_outputs = pd.DataFrame({
    "Output": important_outputs,
    "Predicted value": [y_opt[out_idx[name]] for name in important_outputs]
})

print("\nImportant surrogate-predicted outputs:")
display(opt_outputs)

constraint_summary = pd.DataFrame({
    "Constraint": [
        "|Q1| + |Q2| <= Qmax",
        "MeOH_purity_stream4 >= purity_min",
        "T4 <= T_flash - 2"
    ],
    "Value": [
        best_sol["energy"],
        best_sol["purity"],
        x_opt[idx_T4]
    ],
    "Limit": [
        Qmax,
        purity_min,
        x_opt[idx_Tflash] - 2.0
    ],
    "Slack": [
        best_sol["energy_slack"],
        best_sol["purity_slack"],
        best_sol["T4_slack"]
    ]
})

print("\nConstraint summary:")
display(constraint_summary)



# %%

# CHECK DISTANCE FROM TRAINING DATA

# Find nearest training points in scaled input space
X_train_scaled_all = X_scaler.transform(X_train)
x_opt_scaled = X_scaler.transform(x_opt.reshape(1, -1))

distances = np.linalg.norm(X_train_scaled_all - x_opt_scaled, axis=1)
nearest_idx = np.argsort(distances)[:5]

nearest_rows = []

for idx in nearest_idx:
    row = {
        "distance_scaled": distances[idx],
        "MeOH_recovery_true_dataset": y_train[idx, idx_recovery],
        "MeOH_purity_true_dataset": y_train[idx, idx_purity],
        "Q1_true_dataset": y_train[idx, idx_Q1],
        "Q2_true_dataset": y_train[idx, idx_Q2],
        "energy_true_dataset": abs(y_train[idx, idx_Q1]) + abs(y_train[idx, idx_Q2]),
    }
    
    for j, name in enumerate(input_names_list):
        row[name] = X_train[idx, j]
    
    nearest_rows.append(row)

nearest_table = pd.DataFrame(nearest_rows)

print("Nearest training points to the optimum:")
display(nearest_table)

# %%
# plots for optimization solution interpretation


# Validation comparison on optimization-relevant outputs
opt_outputs = ["Q1_MW", "Q2_MW", "MeOH_recovery", "MeOH_purity_stream4"]
opt_idx = [out_idx[name] for name in opt_outputs]

x = np.arange(len(opt_outputs))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, direct_val_nrmse[opt_idx], width=width, label="Direct NN")
plt.bar(x + width/2, hybrid_val_nrmse[opt_idx], width=width, label="Hybrid Model")

plt.xticks(x, opt_outputs, rotation=20)
plt.ylabel("Validation NRMSE")
plt.title("Validation error comparison on optimization-relevant outputs")
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

#Global selection criterion
model_names = ["Direct NN", "Hybrid Model"]
selection_metric = [
    direct_val_summary["Optimization outputs Mean NRMSE"],
    hybrid_val_summary["Optimization outputs Mean NRMSE"]
]

plt.figure(figsize=(7, 5))
plt.bar(model_names, selection_metric)
plt.ylabel("Validation mean NRMSE\n(optimization outputs)")
plt.title("Surrogate model selection criterion")
plt.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()


# Optimization results: recovery vs energy
energy_all = np.array([s["energy"] for s in solutions])
recovery_all = np.array([s["recovery"] for s in solutions])

categories = []

for s in solutions:
    if s["feasible"]:
        categories.append("Feasible")
    else:
        violations = []
        
        if s["energy_slack"] < -1e-6:
            violations.append("Energy")
        if s["purity_slack"] < -1e-6:
            violations.append("Purity")
        if s["T4_slack"] < -1e-6:
            violations.append("T4")
        
        if len(violations) == 1:
            categories.append(f"{violations[0]} violation")
        else:
            categories.append("Multiple violations")

categories = np.array(categories)

plt.figure(figsize=(9, 6))

plot_styles = {
    "Feasible": {"color": "tab:blue", "marker": "o"},
    "Energy violation": {"color": "tab:red", "marker": "o"},
    "Purity violation": {"color": "tab:orange", "marker": "s"},
    "T4 violation": {"color": "tab:purple", "marker": "^"},
    "Multiple violations": {"color": "tab:gray", "marker": "x"},
}

for cat, style in plot_styles.items():
    mask = categories == cat
    
    if mask.sum() == 0:
        continue
    
    plt.scatter(
        energy_all[mask],
        recovery_all[mask],
        s=65,
        alpha=0.75,
        color=style["color"],
        marker=style["marker"],
        label=cat
    )

plt.axvline(
    Qmax,
    color="black",
    linestyle="--",
    linewidth=2,
    label=r"Energy limit $Q_{max}$"
)

plt.scatter(
    best_sol["energy"],
    best_sol["recovery"],
    s=230,
    marker="*",
    color="gold",
    edgecolor="black",
    linewidth=1.2,
    label="Selected optimum",
    zorder=5
)

plt.xlabel(r"Total energy duty $|Q_1| + |Q_2|$ [MW]")
plt.ylabel("Predicted MeOH recovery [-]")
plt.title("Multi-start constrained optimization results")
plt.grid(True, alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()


# Normalized optimal decision variables within bounds
lower_bounds = np.array([b[0] for b in bounds])
upper_bounds = np.array([b[1] for b in bounds])

x_opt_norm = (x_opt - lower_bounds) / (upper_bounds - lower_bounds)

xpos = np.arange(len(input_names_list))

plt.figure(figsize=(11, 6))

plt.vlines(
    xpos,
    0,
    1,
    linewidth=4,
    alpha=0.5,
    label="Admissible range"
)

plt.scatter(
    xpos,
    x_opt_norm,
    s=90,
    zorder=3,
    label="Optimal value"
)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(1, linestyle="--", linewidth=1)

plt.xticks(xpos, input_names_list, rotation=30)
plt.ylabel("Normalized position within bounds [-]")
plt.title("Optimal decision variables within admissible bounds")
plt.grid(axis="y", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


# %%
# Constraint satisfaction checklist figure

tol = 1e-5

constraint_rows = [
    {
        "Constraint": "Energy duty",
        "Condition": r"$|Q_1| + |Q_2| \leq Q_{max}$",
        "Value": f"{best_sol['energy']:.4f} MW",
        "Limit": f"≤ {Qmax:.4f} MW",
        "Slack": f"{best_sol['energy_slack']:.4f} MW",
        "Status": "Active" if abs(best_sol["energy_slack"]) < tol else
                  "Satisfied" if best_sol["energy_slack"] > 0 else "Violated"
    },
    {
        "Constraint": "Methanol purity",
        "Condition": r"$x_{MeOH,4} \geq x_{min}$",
        "Value": f"{best_sol['purity']:.4f}",
        "Limit": f"≥ {purity_min:.4f}",
        "Slack": f"{best_sol['purity_slack']:.4f}",
        "Status": "Active" if abs(best_sol["purity_slack"]) < tol else
                  "Satisfied" if best_sol["purity_slack"] > 0 else "Violated"
    },
    {
        "Constraint": "HX2 cooling",
        "Condition": r"$T_4 \leq T_{flash} - 2$",
        "Value": f"{x_opt[idx_T4]:.2f} K",
        "Limit": f"≤ {x_opt[idx_Tflash] - 2.0:.2f} K",
        "Slack": f"{best_sol['T4_slack']:.4f} K",
        "Status": "Active" if abs(best_sol["T4_slack"]) < tol else
                  "Satisfied" if best_sol["T4_slack"] > 0 else "Violated"
    }
]

constraint_df = pd.DataFrame(constraint_rows)

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.axis("off")

table = ax.table(
    cellText=constraint_df.values,
    colLabels=constraint_df.columns,
    cellLoc="center",
    colLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)


for j in range(len(constraint_df.columns)):
    cell = table[0, j]
    cell.set_text_props(weight="bold", color="white")
    cell.set_facecolor("#404040")

status_col = list(constraint_df.columns).index("Status")

for i, status in enumerate(constraint_df["Status"], start=1):
    status_cell = table[i, status_col]
    
    if status == "Satisfied":
        status_cell.set_facecolor("#d9ead3")
        status_cell.set_text_props(weight="bold", color="#274e13")
    elif status == "Active":
        status_cell.set_facecolor("#fff2cc")
        status_cell.set_text_props(weight="bold", color="#7f6000")
    else:
        status_cell.set_facecolor("#f4cccc")
        status_cell.set_text_props(weight="bold", color="#990000")

for key, cell in table.get_celld().items():
    cell.set_edgecolor("#bfbfbf")

plt.title(
    "Constraint satisfaction at the selected optimum",
    fontsize=14,
    fontweight="bold",
    pad=18
)

plt.figtext(
    0.5,
    0.02,
    "Positive slack = constraint satisfied with margin; zero slack = active constraint; negative slack = violation.",
    ha="center",
    fontsize=10
)

plt.tight_layout()
plt.show()

# %%
# 3D constraint plot: physical constraint space

from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Extract quantities from all optimization runs
energy_all = np.array([s["energy"] for s in solutions])
purity_all = np.array([s["purity"] for s in solutions])
T4_margin_all = np.array([s["T4_slack"] for s in solutions])
recovery_all = np.array([s["recovery"] for s in solutions])
feasible_all = np.array([s["feasible"] for s in solutions])

# Plot limits with small margins
x_min, x_max = energy_all.min(), energy_all.max()
y_min, y_max = purity_all.min(), purity_all.max()
z_min, z_max = T4_margin_all.min(), T4_margin_all.max()

x_pad = 0.08 * (x_max - x_min + 1e-12)
y_pad = 0.08 * (y_max - y_min + 1e-12)
z_pad = 0.08 * (z_max - z_min + 1e-12)

x_min, x_max = x_min - x_pad, x_max + x_pad
y_min, y_max = y_min - y_pad, y_max + y_pad
z_min, z_max = z_min - z_pad, z_max + z_pad

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")

# -----------------------------
# Constraint boundary planes
# -----------------------------

# Energy plane: |Q1| + |Q2| = Qmax
Y, Z = np.meshgrid(
    np.linspace(y_min, y_max, 15),
    np.linspace(z_min, z_max, 15)
)
X = np.ones_like(Y) * Qmax
ax.plot_surface(X, Y, Z, alpha=0.08, color="tab:blue")

# Purity plane: purity = purity_min
X, Z = np.meshgrid(
    np.linspace(x_min, x_max, 15),
    np.linspace(z_min, z_max, 15)
)
Y = np.ones_like(X) * purity_min
ax.plot_surface(X, Y, Z, alpha=0.08, color="tab:green")

# HX2 cooling plane: T_flash - T4 - 2 = 0
X, Y = np.meshgrid(
    np.linspace(x_min, x_max, 15),
    np.linspace(y_min, y_max, 15)
)
Z = np.zeros_like(X)
ax.plot_surface(X, Y, Z, alpha=0.08, color="tab:orange")

# -----------------------------
# Scatter points
# -----------------------------

# Infeasible points
ax.scatter(
    energy_all[~feasible_all],
    purity_all[~feasible_all],
    T4_margin_all[~feasible_all],
    c="tab:red",
    marker="x",
    s=70,
    alpha=0.75,
    label="Infeasible"
)

# Feasible points, colored by recovery
sc = ax.scatter(
    energy_all[feasible_all],
    purity_all[feasible_all],
    T4_margin_all[feasible_all],
    c=recovery_all[feasible_all],
    cmap="viridis",
    s=90,
    alpha=0.9,
    edgecolor="black",
    linewidth=0.4,
    label="Feasible"
)

# Selected optimum
ax.scatter(
    best_sol["energy"],
    best_sol["purity"],
    best_sol["T4_slack"],
    c="gold",
    marker="*",
    s=450,
    edgecolor="black",
    linewidth=1.2,
    label="Selected optimum",
    zorder=10
)

ax.text(
    best_sol["energy"],
    best_sol["purity"],
    best_sol["T4_slack"],
    f"  Optimum\n  Recovery = {best_sol['recovery']:.3f}",
    fontsize=10,
    fontweight="bold"
)

# -----------------------------
# Labels and formatting
# -----------------------------

ax.set_xlabel(r"Total energy duty $|Q_1| + |Q_2|$ [MW]")
ax.set_ylabel(r"Methanol purity $x_{MeOH,4}$ [-]")
ax.set_zlabel(r"HX2 cooling margin $T_{flash} - T_4 - 2$ [K]")

ax.set_title("Constraint satisfaction and selected optimum", fontsize=14, fontweight="bold")

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_zlim(z_min, z_max)

# View angle chosen to make the three constraint planes visible
ax.view_init(elev=24, azim=-55)

# Colorbar: explains why the optimum was selected among feasible points
cbar = plt.colorbar(sc, ax=ax, shrink=0.65, pad=0.08)
cbar.set_label("Predicted MeOH recovery [-]")

# Custom legend
legend_elements = [
    Line2D([0], [0], marker="o", color="w", label="Feasible",
           markerfacecolor="gray", markeredgecolor="black", markersize=9),
    Line2D([0], [0], marker="x", color="tab:red", label="Infeasible",
           linestyle="None", markersize=9),
    Line2D([0], [0], marker="*", color="w", label="Selected optimum",
           markerfacecolor="gold", markeredgecolor="black", markersize=14),
    Patch(facecolor="tab:blue", alpha=0.15, label=r"Energy limit: $|Q_1|+|Q_2|=Q_{max}$"),
    Patch(facecolor="tab:green", alpha=0.15, label=r"Purity limit: $x_{MeOH,4}=x_{min}$"),
    Patch(facecolor="tab:orange", alpha=0.15, label=r"HX2 limit: $T_{flash}-T_4-2=0$")
]

ax.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(0.0, 1.0))

plt.figtext(
    0.5,
    0.02,
    "A point is feasible only if it lies on the correct side of all three constraint planes. "
    "Among feasible points, the selected optimum has the highest predicted methanol recovery.",
    ha="center",
    fontsize=10
)

plt.tight_layout()
plt.show()


